# Avaliação do Preditor de Preço - Joinville

Análise completa do modelo treinado para prever o valor de imóveis em Joinville.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import sys

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# src/ precisa entrar no path ANTES de qualquer joblib.load: o pipeline
# serializado referencia preprocessador._replace_inf, e sem o modulo o
# unpickle falha. Antes isso so funcionava por ordem de execucao do kernel.
sys.path.insert(0, "../src")


In [ ]:
import datetime
CIDADE = "joinville"

MES_REF = datetime.datetime.now().strftime("%Y-%m")
PASTA_DADOS = Path("../dados") / CIDADE

ARQUIVO_MODELO = PASTA_DADOS / f"{CIDADE}_modelo_geral_{MES_REF}.joblib"
ARQUIVO_OTIM = PASTA_DADOS / f"{CIDADE}_otimizados_melhores_incrementos_{MES_REF}.parquet"
ARQUIVO_TARGET = PASTA_DADOS / f"{CIDADE}_target_transform_{MES_REF}.json"
ARQUIVO_DADOS = PASTA_DADOS / f"{CIDADE}_imoveis_limpo_{MES_REF}.parquet"

print(f"Modelo:    {ARQUIVO_MODELO}")
print(f"Otimizacao: {ARQUIVO_OTIM}")
print(f"Target:    {ARQUIVO_TARGET}")
print(f"Dados:     {ARQUIVO_DADOS}")

## 1. Carregar Melhor Config da Otimização

In [ ]:
df_otim = pd.read_parquet(ARQUIVO_OTIM)
best_idx = df_otim["rmse_otimizado"].idxmin()
best = df_otim.loc[best_idx]

feat_map = json.loads(best.get("feature_transform_map", "{}"))
best_features = list(feat_map.keys())

print(f"Melhor modelo: {best['modelo']}")
print(f"RMSE otimizado: {best['rmse_otimizado']:,.0f}")
print(f"R2 original: {best['r2_original']:.3f}")
print(f"MAPE original: {best['mape_original']:.1f}%")
print(f"MDAPE original: {best['mdape_original']:.1f}%")
print(f"Target transform: {best.get('target_transform', 'none')}")
print(f"Features ({len(best_features)}): {best_features}")
print(f"Tratamento: {best['tratamento']}")
print(f"Scaler: {best['scaler']}")

In [ ]:
best

## 2. Carregar Modelo

In [ ]:
modelo = joblib.load(ARQUIVO_MODELO)
print(f"Modelo carregado: {ARQUIVO_MODELO.name}")
print(f"Tipo: {type(modelo).__name__}")

if Path(ARQUIVO_TARGET).exists():
    with open(ARQUIVO_TARGET) as f:
        meta_target = json.load(f)
    target_transform = meta_target.get("target_transformer", "none")
else:
    target_transform = "none"
print(f"Target transform: {target_transform}")

## 3. Carregar e Preparar Dados

In [ ]:
sys.path.append("../src")
from criando_indices_individuais import CriandoIndicesIndividuais
from funcoes_engenharia_features import engenharia_features_completa, sanitizar_coordenadas

cache_train = PASTA_DADOS / f"{CIDADE}_train_{MES_REF}.parquet"
cache_test = PASTA_DADOS / f"{CIDADE}_test_{MES_REF}.parquet"

if cache_train.exists() and cache_test.exists():
    # Caminho honesto: o modelo publicado foi treinado exatamente neste train.
    train = pd.read_parquet(cache_train)
    test = pd.read_parquet(cache_test)
    print("Avaliacao no holdout de producao (cache de train/test).")
    retreinar = False
else:
    print(f"AVISO: cache de train/test ausente para {MES_REF}.")
    print("O modelo publicado ja viu estas linhas; re-splitar NAO cria holdout.")
    print("Retreinando a config para obter um teste honesto.")
    retreinar = True

if retreinar:
    df = pd.read_parquet(ARQUIVO_DADOS)
    print(f"Parquet limpo: {len(df)} linhas, {len(df.columns)} colunas")

    # Mesmos filtros do treino de producao (selecao_modelos_mlflow.carregar_dados),
    # inclusive o teto de metragem: sem ele entram apartamentos de 8.505 m2, que
    # dominam os maiores erros absolutos e nao existem no modelo publicado.
    mask = (
        (df["metragem"] > 10)
        & (df["tipo_imovel"].isin(["casa", "apartamento"]))
        & (df["preco_por_m2"] >= 100)
        & (df["preco_por_m2"] <= 100_000)
    )
    df_modelo = df[mask].dropna(subset=["valor_imovel"]).copy()
    teto_metragem = df_modelo["tipo_imovel"].map({"apartamento": 1_000, "casa": 3_000})
    df_modelo = df_modelo[df_modelo["metragem"] <= teto_metragem]
    print(f"Apos filtros de producao: {len(df_modelo)} linhas")

    # Coordenadas de fallback do geocoder (12,6% da base, a 400-600 km) precisam
    # sair ANTES dos POIs: dist_centro e todos os score_* derivam de lat/lng.
    df_modelo = sanitizar_coordenadas(df_modelo)

    indices = None
    try:
        indices = CriandoIndicesIndividuais(
            cidade="Joinville, Santa Catarina, Brasil", cache_dir=PASTA_DADOS
        )
        df_modelo = indices.calcular_indices(imoveis_df=df_modelo)
        print("Indices de localizacao calculados.")
    except Exception as e:
        print(f"Erro ao calcular indices: {e}. Usando medias do bairro.")

    # ORDEM OBRIGATORIA: split ANTES da engenharia. criar_features_bairro e
    # criar_score_texto usam o ALVO; calcular sobre a base inteira coloca o teste
    # dentro das agregacoes e invalida qualquer metrica posterior.
    from sklearn.model_selection import train_test_split
    train, test = train_test_split(df_modelo, test_size=0.25, random_state=42)
    train, test = engenharia_features_completa(train, test)

# Topicos entram antes de treinar (e antes de prever, no caminho do cache).
if any(c.startswith("componente_") for c in best_features):
    topicos_path = PASTA_DADOS / f"{CIDADE}_topicos_modelo.pkl"
    if topicos_path.exists():
        topicos_data = joblib.load(topicos_path)
        from utils_topicos import aplicar_topicos
        aplicar_topicos(train, topicos_data)
        aplicar_topicos(test, topicos_data)
        print("Topicos aplicados em train/test.")
    else:
        print("Modelo de topicos nao encontrado. Zerando colunas componente_*.")
        for c in best_features:
            if c.startswith("componente_"):
                train[c] = 0.0
                test[c] = 0.0

if retreinar:
    from sklearn.base import clone
    modelo = clone(modelo).fit(train[best_features], np.log1p(train["valor_imovel"].values))
    print("Modelo reajustado no train local; o teste nunca foi visto.")

print(f"Treino: {len(train)} | Teste: {len(test)}")


## 4. Predizer e Calcular Métricas

In [ ]:
X_test = test[best_features].copy()
y_test = test["valor_imovel"].values

y_pred_raw = modelo.predict(X_test)
if target_transform == "log":
    y_pred = np.expm1(y_pred_raw)
else:
    y_pred = y_pred_raw

erro = y_test - y_pred
erro_abs = np.abs(erro)
erro_pct = np.where(y_test != 0, 100 * erro_abs / y_test, np.nan)

test = test.copy()
test["valor_predito"] = y_pred
test["erro"] = erro
test["erro_abs"] = erro_abs
test["erro_percentual"] = erro_pct

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mdape = np.nanmedian(erro_pct)

print(f"Metricas no conjunto de teste ({len(test)} registros):")
print(f"  RMSE:   R$ {rmse:,.0f}")
print(f"  MAE:    R$ {mae:,.0f}")
print(f"  R2:     {r2:.3f}")
print(f"  MDAPE:  {mdape:.1f}%")

## 5. Scatter Plot: Real vs Predito

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(test["valor_imovel"], test["valor_predito"], alpha=0.3, s=10)
max_val = max(test["valor_imovel"].max(), test["valor_predito"].max())
axes[0].plot([0, max_val], [0, max_val], "r--", lw=2, label="Perfeito")
axes[0].set_xlabel("Valor Real (R$)")
axes[0].set_ylabel("Valor Predito (R$)")
axes[0].set_title("Valor Real vs Predito")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(test["valor_predito"], test["erro"], alpha=0.3, s=10)
axes[1].axhline(y=0, color="r", linestyle="--", lw=2)
axes[1].set_xlabel("Valor Predito (R$)")
axes[1].set_ylabel("Residuo (real - predito)")
axes[1].set_title("Residuos vs Previsao")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Histograma de Erros

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(test["erro"], bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(x=0, color="r", linestyle="--", lw=2)
axes[0].set_xlabel("Erro (real - predito)")
axes[0].set_ylabel("Frequencia")
axes[0].set_title("Distribuicao dos Erros")
axes[0].grid(True, alpha=0.3)

axes[1].hist(test["erro_percentual"].dropna(), bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("Erro Percentual (%)")
axes[1].set_ylabel("Frequencia")
axes[1].set_title("Distribuicao do Erro Percentual")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Erro medio: R$ {test['erro'].mean():,.0f}")
print(f"Erro absoluto medio: R$ {test['erro_abs'].mean():,.0f}")
print(f"Mediana erro absoluto: R$ {test['erro_abs'].median():,.0f}")
print(f"95o percentil erro absoluto: R$ {test['erro_abs'].quantile(0.95):,.0f}")

## 7. Metricas por Faixa de Preco

> **Como ler esta tabela.** `rmse`, `mae`, `mape`, `mdape` e `vies` medem o erro
> dentro da faixa. `r2_faixa` **nao e interpretavel**: ao fatiar o alvo, a
> variancia propria de cada faixa fica pequena, entao qualquer erro a torna
> negativa — nao significa modelo quebrado. O vies mediano e a coluna que mostra
> o encolhimento das pontas (positivo nas faixas baratas, negativo nas caras).


In [ ]:
bins = [0, 200_000, 400_000, 600_000, 1_000_000, float("inf")]
labels = ["0-200k", "200k-400k", "400k-600k", "600k-1M", "1M+"]
test["faixa_preco"] = pd.cut(test["valor_imovel"], bins=bins, labels=labels, right=False)

def calc_rmse(grupo):
    return np.sqrt(mean_squared_error(grupo["valor_imovel"], grupo["valor_predito"]))

metricas_faixa = test.groupby("faixa_preco", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "rmse": calc_rmse(g),
        "mae": g["erro_abs"].mean(),
        "mape": np.nanmean(g["erro_percentual"]),
        "mdape": np.nanmedian(g["erro_percentual"]),
        "vies": 100 * np.nanmedian(g["valor_predito"] - g["valor_imovel"]) / np.nanmedian(g["valor_imovel"]),
        # R2 dentro da faixa nao e interpretavel: a variancia propria da faixa e
        # pequena, entao qualquer erro torna a R2 negativa. Fica registrado por
        # completude; a leitura util esta em RMSE, MAE, MAPE/MdAPE e vies.
        "r2_faixa": r2_score(g["valor_imovel"], g["valor_predito"]) if len(g) > 1 else np.nan,
    })
)

# Arredondar POR COLUNA. O .round(0) no DataFrame inteiro arredondava o R2 para
# inteiro (0,93 virava 1,0; 0,40 virava 0,0), inutilizando a coluna.
metricas_faixa["n"] = metricas_faixa["n"].astype(int)
for col in ("rmse", "mae"):
    metricas_faixa[col] = metricas_faixa[col].round(0)
for col in ("mape", "mdape", "vies"):
    metricas_faixa[col] = metricas_faixa[col].round(2)
metricas_faixa["r2_faixa"] = metricas_faixa["r2_faixa"].round(4)

print("Metricas por faixa de preco:")
print(metricas_faixa.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
metricas_faixa[["rmse", "mae"]].plot(kind="bar", ax=ax)
ax.set_ylabel("Erro (R$)")
ax.set_title("RMSE e MAE por Faixa de Preco")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


## 8. Analise por Bairro

In [ ]:
def calc_rmse(grupo):
    return np.sqrt(mean_squared_error(grupo["valor_imovel"], grupo["valor_predito"]))

metricas_bairro = test.groupby("bairro").apply(
    lambda g: pd.Series({
        "n": len(g),
        "rmse": calc_rmse(g),
        "mae": g["erro_abs"].mean(),
        "valor_medio": g["valor_imovel"].mean(),
        "predicao_media": g["valor_predito"].mean(),
        "mdape": np.nanmedian(g["erro_percentual"]),
    })
).round(0)

metricas_bairro = metricas_bairro[metricas_bairro["n"] >= 10]
metricas_bairro = metricas_bairro.sort_values("rmse")

print(f"Bairros com >= 10 registros: {len(metricas_bairro)}")
print(f"\nTop 10 bairros (menor RMSE):")
print(metricas_bairro.head(10).to_string())
print(f"\nTop 10 piores bairros (maior RMSE):")
print(metricas_bairro.tail(10).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
metricas_bairro.head(20).plot(kind="barh", y="rmse", ax=ax, legend=False)
ax.set_xlabel("RMSE (R$)")
ax.set_title("RMSE por Bairro (Top 20 com menor erro)")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 9. Top 10 Piores Predicoes

In [ ]:
cols_saida = ["titulo", "bairro", "metragem", "quartos", "valor_imovel", "valor_predito", "erro_abs", "erro_percentual"]
cols_existentes = [c for c in cols_saida if c in test.columns]
piores = test.nlargest(10, "erro_abs")[cols_existentes]

print("Top 10 piores predicoes:")
for i, (idx, row) in enumerate(piores.iterrows(), 1):
    titulo = str(row.get("titulo", ""))[:60]
    bairro = row.get("bairro", "N/A")
    metragem = row.get("metragem", "N/A")
    real = row["valor_imovel"]
    predita = row["valor_predito"]
    erro = row["erro_abs"]
    erro_pct = row.get("erro_percentual", 0)
    print(f"\n{i}. {titulo}...")
    print(f"   Bairro: {bairro} | Metragem: {metragem}m2")
    print(f"   Real: R$ {real:,.0f} | Predito: R$ {predita:,.0f} | Erro: R$ {erro:,.0f} ({erro_pct:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
piores_graf = piores.copy()
piores_graf["label"] = piores_graf.apply(
    lambda r: f"{str(r.get('bairro', ''))[:15]} (R${r['valor_imovel']/1000:.0f}k)", axis=1
)
piores_graf.plot(kind="barh", x="label", y=["valor_imovel", "valor_predito"], ax=ax)
ax.set_xlabel("Valor (R$)")
ax.set_title("Top 10 Piores Predicoes")
ax.legend(["Real", "Predito"])
plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

X_test_filtered = test[[c for c in best_features if c in test.columns]]

# O pipeline prediz o LOG do preco. Passar o alvo em R$ faz o baseline sair
# r2 = -1,57: embaralhar qualquer coluna nao muda nada e a importancia fica ~0
# para TODAS as 42 features (foi o que aconteceu na versao anterior). O scoring
# tem de ser na moeda de saida do modelo.
y_log = np.log1p(test["valor_imovel"].values)
print(f"R2 (log) do modelo no teste: {modelo.score(X_test_filtered, y_log):.4f}")
result = permutation_importance(
    modelo, X_test_filtered, y_log, n_repeats=10, random_state=42
)
importance = pd.Series(result.importances_mean, index=X_test_filtered.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importance.plot(kind="barh", ax=ax)
ax.set_xlabel("Importancia (Permutation, queda de R2 no log)")
ax.set_title("Feature Importance")
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 11. QQ Plot e Boxplot dos Residuos

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

residuos = test["erro"].dropna()
stats.probplot(residuos, dist="norm", plot=axes[0])
axes[0].set_title("QQ Plot dos Residuos")
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(residuos, vert=True)
axes[1].set_ylabel("Residuo (real - predito)")
axes[1].set_title("Boxplot dos Residuos")
axes[1].axhline(y=0, color="r", linestyle="--", lw=2)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("Estatisticas dos residuos:")
print(f"  Media: R$ {residuos.mean():,.0f}")
print(f"  Mediana: R$ {residuos.median():,.0f}")
print(f"  Desvio padrao: R$ {residuos.std():,.0f}")
print(f"  Assimetria: {residuos.skew():.4f}")
print(f"  Curtose: {residuos.kurtosis():.4f}")

stat, p_value = stats.shapiro(residuos.sample(min(5000, len(residuos)), random_state=42))
print(f"\nTeste de Shapiro-Wilk:")
print(f"  Estatistica: {stat:.4f}")
print(f"  p-value: {p_value:.6f}")
print(f"  Normal: {'Sim' if p_value > 0.05 else 'Nao'} (alpha=0.05)")

## 12. Resumo Final

In [ ]:
print("=" * 60)
print("RESUMO DA AVALIACAO DO PREDITOR DE PRECO")
print("=" * 60)
print(f"\nModelo: {best['modelo']}")
print(f"Features: {len(best_features)}")
print(f"Target transform: {target_transform}")
print(f"\nMetricas no conjunto de teste ({len(test)} registros):")
print(f"  RMSE:   R$ {rmse:,.0f}")
print(f"  MAE:    R$ {mae:,.0f}")
print(f"  R2:     {r2:.3f}")
print(f"  MDAPE:  {mdape:.1f}%")
print(f"\nDistribuicao por faixa:")
print(test["faixa_preco"].value_counts().sort_index().to_string())
print(f"\n{'=' * 60}")

## 13. Diagnósticos de regressão (correlação, heteroskedasticidade, outliers, alavanca, colinearidade)


### Q1 — Correlação entre os erros


In [ ]:
from statsmodels.stats.stattools import durbin_watson

print("Durbin-Watson:", round(float(durbin_watson(test["erro"].dropna())), 3),
      "(~2 = sem autocorrelação serial; ordem do teste é aleatória, ver texto)")
# O teste que importa p/ imóveis: agrupamento espacial por bairro
g = test.groupby("bairro")["erro"].mean()
print(f"SD médias por bairro: R$ {g.std():,.0f} | SD geral: R$ {test['erro'].std():,.0f}")
print(f"Proporção espacial: {g.std() / test['erro'].std():.0%}")
g.sort_values().head(10).plot(kind="barh", figsize=(8, 5), title="Resíduo médio por bairro (extremos)")
plt.axvline(0, color="black"); plt.tight_layout(); plt.show()


### Q2 — Heteroskedasticidade (variância não-constante)


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

Xw = sm.add_constant(pd.DataFrame({
    "log_pred": np.log1p(test["valor_predito"]),
    "metragem": pd.to_numeric(test["metragem"], errors="coerce")}).fillna(0))
bp = het_breuschpagan(test["erro"].fillna(0).values, Xw.values)
print(f"Breusch-Pagan p-valor: {bp[1]:.6f} (<0.05 = variância não-constante)")

test["__q"] = pd.qcut(test["valor_predito"], 5, duplicates="drop")
print(test.groupby("__q", observed=True)["erro"].std().round(0).to_string())
test = test.drop(columns="__q")

plt.scatter(test["valor_predito"], np.sqrt(np.abs(test["erro"] / test["erro"].std())),
            s=5, alpha=0.3)
plt.xlabel("Valor predito (R$)"); plt.ylabel("sqrt(|resíduo padronizado|)")
plt.title("Scale-Location: dispersão cresce com o preço?"); plt.show()


### Q3 — Outliers

> **Aviso de método.** A OLS abaixo usa `preco_por_m2` = `valor_imovel / metragem`
> como regressor e `valor_imovel` como resposta: o alvo aparece dos dois lados.
> Cook, hat e |t| descrevem a reta de referência, **não** o GradientBoosting.
> Serve como peneira de higiene de dados; não ler como diagnóstico do modelo.


In [ ]:
import statsmodels.api as sm
Xo = sm.add_constant(pd.DataFrame(
    {"m": pd.to_numeric(test["metragem"], errors="coerce"),
     "p": pd.to_numeric(test["preco_por_m2"], errors="coerce")}).fillna(0))
ols = sm.OLS(pd.to_numeric(test["valor_imovel"], errors="coerce"), Xo, missing="drop").fit()
infl = ols.get_influence()
stud = pd.Series(infl.resid_studentized_internal, index=test.index[:len(infl.resid_studentized_internal)])
test["outlier"] = (stud.abs() > 3).reindex(test.index).fillna(False)
print(f"Outliers |t|>3: {int(test['outlier'].sum())} ({test['outlier'].mean():.1%})")
cols = ["url", "titulo", "bairro", "metragem", "valor_imovel", "valor_predito", "erro_percentual"]
display(test.loc[test["outlier"], cols].sort_values("erro_percentual", ascending=False).head(20))


### Q4 — Pontos de alavanca (leverage/Cook)


In [ ]:
hat = pd.Series(infl.hat_matrix_diag, index=test.index[:len(infl.hat_matrix_diag)])
cook = pd.Series(infl.cooks_distance[0], index=test.index[:len(infl.cooks_distance[0])])
test["hat"], test["cook"] = hat.reindex(test.index), cook.reindex(test.index)
print(f"hat máx: {test['hat'].max():.3f} | Cook máx: {test['cook'].max():.3f} | "
      f"Cook>4/n: {int((test['cook'] > 4 / len(test)).sum())}")
plt.scatter(test["hat"], test["erro"].abs(), s=6, alpha=0.4)
plt.xlabel("Alavanca (hat)"); plt.ylabel("|erro|"); plt.title("Leverage × resíduo"); plt.show()
display(test.nlargest(10, "cook")[["url", "titulo", "bairro", "metragem",
      "valor_imovel", "valor_predito", "cook"]])


### Q5 — Colinearidade (VIF com colunas dinâmicas)


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Nomes vêm do artefato do treino (mesma fonte que o modelo usa) —
# se as features mudarem no próximo treino, a célula acompanha sozinha.
feat_info = json.load(open(PASTA_DADOS / f"{CIDADE}_features_modelo_{MES_REF}.json",
                           encoding="utf-8"))
num_cols = [c for c in feat_info.get("features_numericas", []) if c in test.columns]
print(f"Numéricas do modelo: {len(num_cols)} (features_modelo_{MES_REF}.json)")

Xv = (test[num_cols].apply(pd.to_numeric, errors="coerce")
      .fillna(test[num_cols].median(numeric_only=True)).assign(const=1))
vif = sorted(((c, float(variance_inflation_factor(Xv.values, i)))
              for i, c in enumerate(Xv.columns) if c != "const"),
             key=lambda t: -t[1])
print("Top VIF (inf = colinearidade exata):")
for c, v in vif[:10]:
    print(f"  {c}: {'inf' if v == float('inf') else round(v, 1)}")


# Predizer precos para os dados limpos

In [ ]:
df_limpo = pd.read_parquet(PASTA_DADOS / f"joinville_imoveis_limpo_{MES_REF}.parquet")
print(f"Parquet limpo: {len(df_limpo)} linhas, {len(df_limpo.columns)} colunas")

antes = len(df_limpo)
df_limpo = df_limpo[
    (df_limpo["valor_imovel"] > 0) &
    (df_limpo["valor_imovel"] <= 6_000_000)
]
print(f"Filtro outliers (<=6M): {antes} -> {len(df_limpo)} ({antes - len(df_limpo)} removidos)")

In [ ]:
indices_full = None
try:
    indices_full = CriandoIndicesIndividuais(cidade="Joinville, Santa Catarina, Brasil", cache_dir=PASTA_DADOS)
    df_limpo = indices_full.calcular_indices(imoveis_df=df_limpo)
except Exception as e:
    print(f"Erro ao calcular indices: {e}")

df_limpo, _ = engenharia_features_completa(df_limpo, df_limpo.copy())

if any(c.startswith("componente_") for c in best_features):
    topicos_path = PASTA_DADOS / f"{CIDADE}_topicos_modelo.pkl"
    if topicos_path.exists():
        topicos_data = joblib.load(topicos_path)
        from utils_topicos import aplicar_topicos
        aplicar_topicos(df_limpo, topicos_data)
        print("Topicos aplicados ao dataset completo.")
    else:
        print("Modelo de topicos nao encontrado. Zerando colunas componente_*.")
        for c in best_features:
            if c.startswith("componente_"):
                df_limpo[c] = 0.0

for col in best_features:
    if col not in df_limpo.columns:
        if col in ["bairro_cluster", "novo_lancamento", "tem_elevador", "sem_rua"]:
            df_limpo[col] = 0

X_pred = df_limpo[[c for c in best_features if c in df_limpo.columns]]
y_pred_raw = modelo.predict(X_pred)

if target_transform == "log":
    df_limpo["valor_predito"] = np.expm1(y_pred_raw)
else:
    df_limpo["valor_predito"] = y_pred_raw

print(f"valor_predito adicionada ({len(df_limpo)} registros)")
print(f"Media: R$ {df_limpo['valor_predito'].mean():,.0f}")
print(f"Mediana: R$ {df_limpo['valor_predito'].median():,.0f}")
print(f"Min: R$ {df_limpo['valor_predito'].min():,.0f}")
print(f"Max: R$ {df_limpo['valor_predito'].max():,.0f}")

In [23]:
# df_limpo.to_parquet(PASTA_DADOS / f"joinville_imoveis_limpo_{MES_REF}.parquet")